# 双色球多算法研究（未发版功能）

同一份数据上依次跑：概率与显著性分析 → 多算法对照回测 → 可视化与单文件 HTML 仪表盘。

**本 Notebook 不承诺预测。** 每个算法都与随机基线在同窗口、同成本下逐期对比，每个排名都附 95% 置信区间与卡方检验结论。

如果某个算法看起来赢了，请先看它的 `distinct_blue` 与 `blue_entropy`：输出退化成常数时，命中率看起来会“稳定”，但那不是预测。

In [ ]:
# ===== 用户配置 =====
SEED = None          # None = 每次运行随机生成并打印；要复现就把打印出的 SEED 填回来
WINDOW = 100         # 回测的信息窗口（期）
MAX_PERIODS = 600    # 回测期数上限，0 表示全部（全量较慢）
RUN_ARIMA = False    # 需要 statsmodels
RUN_WEBAPP = False   # 需要 gradio，打开后以 share=True 对外发布
REPO_URL = 'https://github.com/wangmou23/ssq-quant-research-system.git'
REPO_DIR = '/kaggle/working/ssq-quant-research-system'

import time
if SEED is None:
    SEED = int(time.time_ns() % (2**31))
print(f'实际使用 SEED={SEED}（本次运行以此为准；要复现请把该值填回 SEED）')

In [ ]:
# ===== 加载源码 =====
import os, sys, subprocess, pathlib, shutil
repo_dir = pathlib.Path(REPO_DIR)
if repo_dir.exists() and not (repo_dir / 'ssq_research').exists():
    shutil.rmtree(repo_dir)
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
else:
    # 目录已存在（重跑，或上一个 Notebook 克隆到同一路径）时必须强制对齐远端，
    # 否则会静默复用旧代码：跑出来的结果和仓库里看到的不是同一份。
    fetched = subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth', '1', 'origin', 'main'],
                             capture_output=True, text=True)
    if fetched.returncode == 0:
        subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'FETCH_HEAD'], check=True)
        print('已把已有目录更新到远端 main')
    else:
        print('警告：已有目录无法更新，继续用现有代码 ->', fetched.stderr.strip()[:200])
sys.path.insert(0, str(repo_dir))
print('提交:', subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], text=True).strip())
import ssq_research
print('版本:', ssq_research.__version__)


In [ ]:
# ===== 依赖检查 =====
# 不执行 pip install -e：仓库已修好打包配置，但这里只需 sys.path 引入，不必装包。
# 注意：available_algorithms() 会真的 import tensorflow / statsmodels 去探测，
# Kaggle 上首次约 10~30 秒，属正常现象，不是卡住。
import importlib.util
for name in ('numpy', 'matplotlib', 'statsmodels', 'tensorflow', 'gradio'):
    print(f'{name:14s}', 'OK' if importlib.util.find_spec(name) else 'MISSING')
from ssq_research.scores import available_algorithms
print('算法可用性:', available_algorithms())
print('注：lstm 为 True 只表示 TensorFlow 可导入；本 Notebook 不训练 LSTM，'
      'LSTM 回测在 ssq_lstm_kaggle.ipynb 里跑。')


In [ ]:
# ===== 取数 =====
from datetime import datetime, timezone
from ssq_research.data_fetch import fetch_ssq_draws, DEFAULT_SSQ_URL
try:
    draws = fetch_ssq_draws(DEFAULT_SSQ_URL, timeout=30)
    print('数据源:', DEFAULT_SSQ_URL)
except Exception as exc:
    raise RuntimeError('自动取数失败，请确认 Kaggle Internet=On') from exc
print('抓取时间 UTC:', datetime.now(timezone.utc).isoformat())
print('期数:', len(draws), '范围:', draws[0].issue, '→', draws[-1].issue)

In [ ]:
# ===== 概率与显著性分析 =====
from ssq_research.stats import build_probability_report
report = build_probability_report(draws)
print(f'样本 {report.samples} 期：{report.first_issue} → {report.last_issue}')
print(f'红球 卡方 χ²={report.red_uniformity.statistic:.2f} df={report.red_uniformity.degrees_of_freedom} p={report.red_uniformity.p_value:.4f}')
print(f'蓝球 卡方 χ²={report.blue_uniformity.statistic:.2f} df={report.blue_uniformity.degrees_of_freedom} p={report.blue_uniformity.p_value:.4f}')
print()
print('红球 Top5（括号内为相对理论值 18.18% 的偏差，[偏离] 表示超出单球 95% 区间）：')
for rank, row in enumerate(report.red_top, start=1):
    low, high = row.interval
    flag = ' [偏离]' if row.deviates_significantly else ''
    print(f'  {rank}. 红球 {row.number:02d}  {row.count:4d} 次  {row.rate*100:6.2f}%  (偏差 {row.deviation*100:+.2f}%, 95% 区间 {low*100:.2f}%~{high*100:.2f}%){flag}')
print()
print('蓝球 Top5（理论值 6.25%）：')
for rank, row in enumerate(report.blue_top, start=1):
    low, high = row.interval
    flag = ' [偏离]' if row.deviates_significantly else ''
    print(f'  {rank}. 蓝球 {row.number:02d}  {row.count:4d} 次  {row.rate*100:6.2f}%  (偏差 {row.deviation*100:+.2f}%, 95% 区间 {low*100:.2f}%~{high*100:.2f}%){flag}')
print()
print('各奖级理论概率：')
for odds in report.prizes:
    print(f'  {odds.level}  1 / {odds.one_in:,.0f}')
print()
if report.has_detectable_pattern:
    print('结论：至少一类球偏离均匀假设，请先排查数据完整性（漏期 / 错列 / 重复期号）再解读排名。')
else:
    print('结论：两类球都与公平开奖一致，上面的排名只描述历史，不含对下一期的预测信息。')

In [ ]:
# ===== 多算法对照回测（同窗口、同成本、同一随机基线）=====
from ssq_research.core import RandomStrategy, FrequencyStrategy, run_backtest, summarize, blue_degeneracy_report
from ssq_research.cli import algorithm_strategies

strategies = [RandomStrategy(), FrequencyStrategy()] + algorithm_strategies(include_arima=RUN_ARIMA)
ordered = sorted(draws, key=lambda d: (d.draw_date, d.issue))
if MAX_PERIODS > 0:
    ordered = ordered[-(MAX_PERIODS + WINDOW):]
print(f'回测期数上限 {MAX_PERIODS or "全部"}，窗口 {WINDOW}，策略数 {len(strategies)}')
rows = run_backtest(ordered, strategies, window=WINDOW, seed=SEED)

print()
print(f'{"策略":<24}{"期数":>6}{"红均":>8}{"蓝均":>8}{"蓝去重":>8}{"蓝熵":>8}{"净收益":>10}')
for item in summarize(rows):
    net = item.get('net')
    net_text = '未知' if net is None else f'{net:.1f}'
    print(f'{item["strategy"]:<24}{item["trials"]:>6}{item["avg_red_hits"]:>8.3f}{item["avg_blue_hits"]:>8.4f}{item.get("distinct_blue", 0):>8}{item.get("blue_entropy", 0.0):>8.3f}{net_text:>10}')
print()
print('退化检测（distinct_blue 长期只有 1~3、熵远低于 4 bit 即说明输出已退化为常数）：')
for name in sorted({row['strategy'] for row in rows}):
    group = [row for row in rows if row['strategy'] == name]
    print(' ', name, blue_degeneracy_report(group))

In [ ]:
# ===== 生成单文件 HTML 仪表盘 =====
from ssq_research.report import build_report_html
from IPython.display import FileLink, display

# 第一个参数必须传全量 draws，不能传上面回测用的 ordered（那是按 MAX_PERIODS 截断的切片）：
# 报告内部会自己重算频次表和卡方，传切片会让 HTML 里的样本数变成 700，
# 与上面 cell 5 打印的数字对不上，看起来像两份不同的数据。
document = build_report_html(draws, rows, seed=SEED, window=WINDOW)

report_path = pathlib.Path('/kaggle/working') / 'ssq_report.html'
if not report_path.parent.exists():          # 非 Kaggle 环境直接落到当前目录
    report_path = pathlib.Path('ssq_report.html')
report_path.write_text(document, encoding='utf-8')
print('报告已生成:', report_path.resolve(), f'{report_path.stat().st_size / 1024:.1f} KB')
print(f'报告中频次/卡方基于全量 {len(draws)} 期；多算法对照表基于最近 {MAX_PERIODS} 期回测。')
print('图中标签为英文：Kaggle 的 matplotlib 默认字体没有中文字形，中文会显示成方框。')
display(FileLink(str(report_path)))


In [ ]:
# ===== 可选：实时交互界面（Gradio）=====
if RUN_WEBAPP:
    from ssq_research.webapp import launch
    launch(draws, share=True)
else:
    print('RUN_WEBAPP=False：需要浏览器里实时查询时改为 True（依赖 gradio）。')
    print('Kaggle 不暴露端口，所以必须 share=True 才能访问；静态报告是更稳的产物。')

## 结果解释与边界

- **排名 ≠ 预测。** 高频号码描述的是历史。要判断“热号”是否有意义，看它是否超出 95% 区间；即使超出，同时检验 33 个号时出现一两个这种情况本身就在期望之内。
- **卡方 p 值**：p 大说明历史分布与均匀一致。这时任何“冷热号”结论都只是噪声。
- **集成学习不提高中奖概率。** 它降低的是多个模型输出之间的分歧，不是开奖本身的随机性。多个都无法超越随机基线的模型，加权平均后依然无法超越。
- **净收益里的“未知”**：一/二等奖金额依赖奖池，不能当作 0 元计入，否则中头奖反而显示为亏损。
- **同比才有意义**：任何策略都必须与同票数、同成本的随机基线比较；单次命中或最高奖级不能作为优势证据。

一等奖概率恒为 1 / 17,721,088，与选号方式无关。这是本项目所有实验的前提。